In [1]:
#Import dependencies
import requests
from bs4 import BeautifulSoup
import pandas as pd
import random
import time

In [2]:
jobs_title = [
    #Data
    "Data Analyst",
    "Data Engineer",
    "Analytics Engineer",
    "Machine Learning Engineer",
    
    # 3 משרות פיתוח (Software Development)
    "Python Developer",
    "Full Stack Developer",
    "Backend Developer",
    
    # 2 משרות QA
    "QA Engineer",
    "Automation Engineer",
    
    # 1 משרה של מנהל מוצר
    "Product Manager"
]  # Job title
location = "Israel"  # Job location
start = 0  # Starting point for pagination
end = 50

In [3]:
#Create an empty list to store the job postings
id_list = []


In [4]:
  
for start in range(0,400,10):

    # Construct the URL for LinkedIn job search
    list_url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?location={location}&f_E=2%2C1%2C3&start={start}&f_TPR=r86400&pageNum=0"

    # Send a GET request to the URL and store the response
    response = requests.get(list_url)

    #Get the HTML, parse the response and find all list items(jobs postings)
    list_data = response.text
    list_soup = BeautifulSoup(list_data, "html.parser")
    page_jobs = list_soup.find_all("li")
    if not page_jobs:
        break
    for job in page_jobs:
        base_card_div = job.find("div", {"class": "base-card"})
        if not base_card_div:
            break
        job_id = base_card_div.get("data-entity-urn").split(":")[3]
        id_list.append(job_id)
    print(f"{len(id_list)} jobs extracted from LinkedIn")
    

    time.sleep(random.uniform(0.5,2))        
        

10 jobs extracted from LinkedIn
20 jobs extracted from LinkedIn
30 jobs extracted from LinkedIn
40 jobs extracted from LinkedIn
50 jobs extracted from LinkedIn
60 jobs extracted from LinkedIn
70 jobs extracted from LinkedIn
80 jobs extracted from LinkedIn
90 jobs extracted from LinkedIn
100 jobs extracted from LinkedIn
110 jobs extracted from LinkedIn
120 jobs extracted from LinkedIn
130 jobs extracted from LinkedIn
140 jobs extracted from LinkedIn
150 jobs extracted from LinkedIn
160 jobs extracted from LinkedIn
170 jobs extracted from LinkedIn
180 jobs extracted from LinkedIn
190 jobs extracted from LinkedIn
200 jobs extracted from LinkedIn
210 jobs extracted from LinkedIn
220 jobs extracted from LinkedIn
230 jobs extracted from LinkedIn
240 jobs extracted from LinkedIn
250 jobs extracted from LinkedIn
260 jobs extracted from LinkedIn
269 jobs extracted from LinkedIn
279 jobs extracted from LinkedIn
289 jobs extracted from LinkedIn
299 jobs extracted from LinkedIn
302 jobs extracted 

In [5]:
id_list

['4370963330',
 '4383805162',
 '4382820819',
 '4383724471',
 '4214614545',
 '4382466446',
 '4383786827',
 '4382561327',
 '4383835754',
 '4384103280',
 '4382091725',
 '4383796982',
 '4381514993',
 '4382480812',
 '4383840707',
 '4377871605',
 '4382076250',
 '4384175338',
 '4383308880',
 '4382502573',
 '4351762567',
 '4382473162',
 '4383837784',
 '4382893514',
 '4384116842',
 '4383836454',
 '4383844464',
 '4232901222',
 '4384193582',
 '4384149706',
 '4366369464',
 '4382896536',
 '4364671319',
 '4381608552',
 '4382557972',
 '4383800946',
 '4382520253',
 '4383758388',
 '4382570766',
 '4384308062',
 '4382570766',
 '4384102322',
 '4375519661',
 '4372846093',
 '4383851043',
 '4375153638',
 '4382534603',
 '4382070221',
 '4384161127',
 '4382532679',
 '4384147311',
 '4384130884',
 '4366367063',
 '4373448873',
 '4382889845',
 '4383203314',
 '4384174666',
 '4382897550',
 '4350253782',
 '4384180970',
 '4381833685',
 '4383785875',
 '4382577115',
 '4324275243',
 '4384179489',
 '4383858048',
 '43824817

In [6]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Referer": "https://www.linkedin.com/jobs/search?keywords=Data%20Analyst", # שינוי Referer
    "Upgrade-Insecure-Requests": "1"
}

# Initialize an empty list to store job information
job_list = []
# Loop through the list of job IDs and get each URL
for job_id in id_list:
    # Construct the URL for each job using the job ID
    
    job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"
    
    # Send a GET request to the job URL and parse the reponse
    try:
        job_response = requests.get(job_url, headers=headers)
        
        if job_response.status_code != 200:
            print(f"Error {job_response.status_code} for job {job_id[0]}")
            if job_response.status_code == 429:
                print("to many requests/ 5 minutes sleep")
                time.sleep(300) # עצירה ל-5 דקות אם נחסמנו
            continue
        print(job_response.status_code)
        job_soup = BeautifulSoup(job_response.text, "html.parser")
        
        # Create a dictionary to store job details
        job_post = {}
        
        # Try to extract and store the job title
        try:
            job_post["job_title"] = job_soup.find("h2", {"class":"top-card-layout__title font-sans text-lg papabear:text-xl font-bold leading-open text-color-text mb-0 topcard__title"}).text.strip()
        except:
            job_post["job_title"] = None
            
        # Try to extract and store the company name
        try:
            job_post["company_name"] = job_soup.find("a", {"class": "topcard__org-name-link topcard__flavor--black-link"}).text.strip()
        except:
            job_post["company_name"] = None
            
        
        # אנחנו מחפשים את ה-span השני בתוך ה-row של ה-flavor
        try:
        # בדרך כלל האינדקס 0 הוא המיקום (במקרה הזה Petah Tikva)
            job_post["location"] = job_soup.find_all("span", {"class": "topcard__flavor topcard__flavor--bullet"})[0].text.strip()
        except:
            job_post["location"] = None
            
        # Try to extract and store the time posted
        try:
            job_post["time_posted"] = job_soup.find("span", {"class": "posted-time-ago__text topcard__flavor--metadata"}).text.strip()
        except:
            job_post["time_posted"] = None
            
        # Try to extract and store the number of applicants
        try:
            job_post["num_applicants"] = job_soup.find("span", {"class": "num-applicants__caption topcard__flavor--metadata topcard__flavor--bullet"}).text.strip()
        except:
            job_post["num_applicants"] = None
        try:
            job_description_div = job_soup.find('div', class_='description__text--rich')
            job_post["description_text"] = job_description_div.get_text(separator='\n', strip=True)
        except:
            job_post["description_text"] = None
            
        job_post["URL"] = job_url
            
        # Append the job details to the job_list
        job_list.append(job_post)

    except Exception as e:
        print(f"Request failed: {e}")
        time.sleep(10)
        continue

    # המתנה ארוכה יותר ומשתנה
    time.sleep(random.uniform(5, 12))
    

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200


In [7]:
#Check if the list contains all the desired data
job_list

[{'job_title': 'Financial Analyst',
  'company_name': 'Deloitte',
  'location': 'Tel Aviv District, Israel',
  'time_posted': None,
  'num_applicants': None,
  'description_text': 'Deloitte is looking for Financial Analyst. Overall responsibility for the management, control, and support of client projects, focusing on the financial aspects of projects and clients, while delivering a range of services in reporting, controls, accounts receivable, and budgeting. The role involves working with senior managers and diverse stakeholders and requires strong attention to detail. Review and control basic pricing prior to submission in the organization’s systems. Update tracking of data and engagement agreements (SOWs). Create a work breakdown structure (WBS) in the SWIFT system and track project financial data. Reconcile project data against SWIFT, monitor account status, and other financial targets. Provide periodic reporting to senior leadership (Partner/Principal/Managing Director—PPMD) on pr

In [8]:
# Create a pandas DataFrame using the list of job dictionaries 'job_list'
jobs_df = pd.DataFrame(job_list)
jobs_df

,job_title,company_name,location,time_posted,num_applicants,description_text,URL
0,Financial Analyst,Deloitte,"Tel Aviv District, Israel",None,None,Deloitte is looking for Financial Analyst. Ove...,https://www.linkedin.com/jobs-guest/jobs/api/j...
1,Sales Development Representative,Hud,"Tel Aviv-Yafo, Tel Aviv District, Israel",None,None,Job Description\nAbout The Role\nHud is lookin...,https://www.linkedin.com/jobs-guest/jobs/api/j...
2,Web Application Developers,GoodWork Environmental Jobs,"Or Akiva, Haifa District, Israel",None,53 applicants,Web application development\nPosition:\nWeb Ap...,https://www.linkedin.com/jobs-guest/jobs/api/j...
3,Client Success Representative,Jobgether,Israel,None,51 applicants,This position is posted by Jobgether on behalf...,https://www.linkedin.com/jobs-guest/jobs/api/j...
4,Junior Technical Support Engineer (fresh gradu...,OPSWAT,"Petah Tikva, Center District, Israel",None,None,"OPSWAT, a global leader in IT, OT, and ICS cri...",https://www.linkedin.com/jobs-guest/jobs/api/j...
...,...,...,...,...,...,...,...
297,Sr/ Principal Backend Developer - Cloud Applic...,Palo Alto Networks,"Tel Aviv-Yafo, Tel Aviv District, Israel",1 day ago,None,"Our Mission\nAt Palo Alto Networks®, we’re uni...",https://www.linkedin.com/jobs-guest/jobs/api/j...
298,SAP SD / OTC Implementer,SQLink Group,"Petah Tikva, Center District, Israel",None,None,We are recruiting an\nSAP SD / OTC Implementer...,https://www.linkedin.com/jobs-guest/jobs/api/j...
299,Experienced Algorithm Engineer for Priority Se...,Mobileye,"Haifa District, Israel",None,None,Join Mobileye's REM (Road Experience Managemen...,https://www.linkedin.com/jobs-guest/jobs/api/j...
300,Senior SW Technical Lead,ForSight Robotics,"Caesarea, Haifa District, Israel",None,None,Who we are\nWe are here to improve lives by re...,https://www.linkedin.com/jobs-guest/jobs/api/j...


In [9]:
from datetime import date

# יצירת תאריך של היום בפורמט YYYY-MM-DD
today = date.today()

# הוספת העמודה ל-DataFrame
jobs_df['search_date'] = today

# הצגת התוצאה
print(jobs_df.head())

                                           job_title  \
0                                  Financial Analyst   
1                   Sales Development Representative   
2                         Web Application Developers   
3                      Client Success Representative   
4  Junior Technical Support Engineer (fresh gradu...   

                  company_name                                  location  \
0                     Deloitte                 Tel Aviv District, Israel   
1                          Hud  Tel Aviv-Yafo, Tel Aviv District, Israel   
2  GoodWork Environmental Jobs          Or Akiva, Haifa District, Israel   
3                    Jobgether                                    Israel   
4                       OPSWAT      Petah Tikva, Center District, Israel   

  time_posted num_applicants  \
0        None           None   
1        None           None   
2        None  53 applicants   
3        None  51 applicants   
4        None           None   

             

In [11]:
jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 302 entries, 0 to 301
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   job_title         302 non-null    object
 1   company_name      302 non-null    object
 2   location          302 non-null    object
 3   time_posted       38 non-null     object
 4   num_applicants    105 non-null    object
 5   description_text  302 non-null    object
 6   URL               302 non-null    object
 7   search_date       302 non-null    object
dtypes: object(8)
memory usage: 19.0+ KB


In [ ]:
import sqlite3
import pandas as pd



# 3. התחברות ל-DB (יוצר את הקובץ אם הוא לא קיים)
conn = sqlite3.connect('linkedin_jobs.db')

# 4. שמירת הנתונים לטבלה בשם 'jobs'
# if_exists='replace' ימחוק את הטבלה הישנה ויצור חדשה עם הנתונים מה-CSV
# אם אתה רוצה להוסיף לקיים, שנה ל-'append'
jobs_df.to_sql('jobs', conn, if_exists='append', index=False)

# 5. סגירת החיבור
conn.close()

In [3]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('linkedin_jobs.db')
df_jobs = pd.read_sql_query("SELECT * FROM jobs", conn)
df_jobs.head()

,job_title,company_name,location,time_posted,num_applicants,description_text,URL,search_date
0,Financial Analyst,Deloitte,"Tel Aviv District, Israel",None,None,Deloitte is looking for Financial Analyst. Ove...,https://www.linkedin.com/jobs-guest/jobs/api/j...,2026-03-11
1,Sales Development Representative,Hud,"Tel Aviv-Yafo, Tel Aviv District, Israel",None,None,Job Description\nAbout The Role\nHud is lookin...,https://www.linkedin.com/jobs-guest/jobs/api/j...,2026-03-11
2,Web Application Developers,GoodWork Environmental Jobs,"Or Akiva, Haifa District, Israel",None,53 applicants,Web application development\nPosition:\nWeb Ap...,https://www.linkedin.com/jobs-guest/jobs/api/j...,2026-03-11
3,Client Success Representative,Jobgether,Israel,None,51 applicants,This position is posted by Jobgether on behalf...,https://www.linkedin.com/jobs-guest/jobs/api/j...,2026-03-11
4,Junior Technical Support Engineer (fresh gradu...,OPSWAT,"Petah Tikva, Center District, Israel",None,None,"OPSWAT, a global leader in IT, OT, and ICS cri...",https://www.linkedin.com/jobs-guest/jobs/api/j...,2026-03-11


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
try:
    nlp = spacy.load('en_core_web_md')
except:
    spacy.cli.download("en_core_web_md")
    nlp = spacy.load('en_core_web_md')

In [ ]:
df_jobs['embedding'] = None
for  row in jobs_df.iterrows():
        job_desc = row['description_text']
        job_emb = model.encode(job_desc.lower(), convert_to_tensor=True)
        row['embedding'] = job_emb
       
    
   